# 00. Thu thập & Ghép dữ liệu thô (Data Collection & Merge)

**Các bước chính:**
1. Đọc JSON review, làm phẳng (flatten) cấu trúc lồng nhau thành DataFrame.
2. Làm sạch: loại trùng lặp, loại dòng thiếu `review_text`, chuẩn hoá
   `review_time` → `review_year`/`review_month`.
3. Đọc CSV sản phẩm, loại cột không dùng, lọc về 5 brand mục tiêu
   (CeraVe, Eucerin, Murad, Obagi, URIAGE)
4. Ghép (left join) 2 bảng theo `product_id`.

**Output:** `data/skincare_product.csv` (7.806 dòng, 10 cột) — input cho
`01_data_processing.ipynb`.

## 1. Đọc & làm phẳng dữ liệu review (JSON)

In [ ]:
import json
import pandas as pd

RAW_DIR = "../data/raw"  

with open(f"{RAW_DIR}/lazada_reviews.json", "r", encoding="utf-8") as f:
    data = json.load(f)

if isinstance(data, dict):
    data = [data]

all_reviews = []
for item in data:
    item_id = item.get("itemId")
    crawled_at = item.get("crawledAt")
    by_stars = item.get("byStars", {})
    for star_level, star_data in by_stars.items():
        for rev in star_data.get("reviews", []):
            all_reviews.append({
                "itemId": item_id,
                "crawledAt": crawled_at,
                "star_level_group": star_level,
                "user_id": rev.get("user_id"),
                "product_id": rev.get("product_id"),
                "review_time": rev.get("review_time"),
                "review_helpfulness": rev.get("review_helpfulness"),
                "review_score": rev.get("review_score"),
                "review_text": rev.get("review_text"),
            })

df_reviews = pd.DataFrame(all_reviews)
print("Kích thước bảng review:", df_reviews.shape)

Kích thước bảng review: (24177, 9)


## 2. Làm sạch bảng review

In [2]:
# Loại trùng lặp tuyệt đối
df_reviews = df_reviews.drop_duplicates()

# Loại dòng thiếu review_text
df_reviews = df_reviews.dropna(subset=["review_text"])

# Loại các cột không dùng cho phân tích
df_reviews = df_reviews.drop(
    ["itemId", "crawledAt", "user_id", "star_level_group", "review_helpfulness"], axis=1
)

# Chuẩn hoá thời gian: loại các dòng review_time dạng tương đối ("x days ago"...)
df_reviews = df_reviews[
    ~df_reviews["review_time"].astype(str).str.contains(r"\bago\b", case=False, na=False)
].copy()
df_reviews["review_date"] = pd.to_datetime(df_reviews["review_time"])
df_reviews["review_year"] = df_reviews["review_date"].dt.year
df_reviews["review_month"] = df_reviews["review_date"].dt.month
df_reviews = df_reviews.drop(["review_time", "review_date"], axis=1)

df_reviews.info()

<class 'pandas.core.frame.DataFrame'>
Index: 23327 entries, 0 to 24176
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   product_id    23327 non-null  int64 
 1   review_score  23327 non-null  int64 
 2   review_text   23327 non-null  object
 3   review_year   23327 non-null  int32 
 4   review_month  23327 non-null  int32 
dtypes: int32(2), int64(2), object(1)
memory usage: 911.2+ KB


## 3. Đọc & làm sạch bảng sản phẩm (CSV)

In [3]:
df_products = pd.read_csv(f"{RAW_DIR}/pages-new.csv")

df_products = df_products.drop(
    ["brand_id", "seller_id", "review_count", "seller_name", "location",
     "in_stock", "rating_score", "product_url"],
    axis=1
)

# Lọc về 5 brand mục tiêu của project
target_brands = ["CeraVe", "Eucerin", "Murad", "Obagi", "URIAGE"]
df_products = df_products[df_products["brand_name"].isin(target_brands)].reset_index(drop=True)

# Điền price_original còn thiếu bằng price_current (không giảm giá)
df_products["price_original"] = df_products["price_original"].fillna(df_products["price_current"])

print(df_products["brand_name"].value_counts())
df_products.info()

brand_name
Eucerin    57
Obagi      47
URIAGE     39
CeraVe     32
Murad      31
Name: count, dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 206 entries, 0 to 205
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   product_id      206 non-null    int64  
 1   product_name    206 non-null    object 
 2   brand_name      206 non-null    object 
 3   price_current   206 non-null    int64  
 4   price_original  206 non-null    float64
 5   sold_count      206 non-null    int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 9.8+ KB


## 4. Ghép 2 bảng (left join theo product_id)

In [4]:
df = pd.merge(df_products, df_reviews, on="product_id", how="left")
df = df.dropna(subset=["review_text"]).reset_index(drop=True)
df = df.drop_duplicates()

print(df.shape)
df.info()

(7806, 10)
<class 'pandas.core.frame.DataFrame'>
Index: 7806 entries, 0 to 8074
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   product_id      7806 non-null   int64  
 1   product_name    7806 non-null   object 
 2   brand_name      7806 non-null   object 
 3   price_current   7806 non-null   int64  
 4   price_original  7806 non-null   float64
 5   sold_count      7806 non-null   int64  
 6   review_score    7806 non-null   float64
 7   review_text     7806 non-null   object 
 8   review_year     7806 non-null   float64
 9   review_month    7806 non-null   float64
dtypes: float64(4), int64(3), object(3)
memory usage: 670.8+ KB


## 5. Lưu kết quả

In [5]:
df.to_csv("../data/skincare_product.csv", index=False)
print(f"Đã lưu ../data/skincare_product.csv ({df.shape[0]} dòng, {df.shape[1]} cột)")

Đã lưu ../data/skincare_product.csv (7806 dòng, 10 cột)
